In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ========================== CELL 1: INSTALL & LOGIN ==========================
!pip install -q transformers accelerate bitsandbytes torch pandas tqdm

from huggingface_hub import login
login()  # ← Paste your Hugging Face token (needs "read" access to meta-llama)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import pandas as pd
from tqdm import tqdm
import os



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
# ========================== CELL 2: LOAD LLAMA-3.2-3B-INSTRUCT (4-bit) ==========================
model_name = "meta-llama/Llama-3.2-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
# ========================== CELL 3: DEFINE 3 LLM AGENTS ==========================
def query_agent(system_prompt, user_message, max_new=150):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]

    try:
        # Preferred way – ask for dict explicitly
        encoded = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True
        )
        input_ids = encoded["input_ids"].to(model.device)

    except (KeyError, TypeError, ValueError):
        # Fallback: render to string → tokenize manually
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False
        )
        encoded = tokenizer(
            prompt_text,
            return_tensors="pt",
            add_special_tokens=False
        )
        input_ids = encoded["input_ids"].to(model.device)

    with torch.no_grad():  # good practice
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new,
            do_sample=False,
            temperature=0.0,   # deterministic → remove temperature/top_p warning
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][input_ids.shape[1]: ],
        skip_special_tokens=True
    )
    return response.strip()

# ==================== AGENT 1: Fact Checker (real / fake(misinformation) / unrelated) ====================
fact_system = """You are FactCheckerAgent.
Analyze the review text against the title and URL (if present).
Classify ONLY as:
- real → genuine review about the product/service
- fake(misinformation) → false claims, spam, fabricated info
- unrelated → off-topic, not about the product

If URL and title are None/blank, base decision on whether text looks like a normal review comment.

Output EXACTLY this JSON and nothing else:
{"label": "real"} or {"label": "fake(misinformation)"} or {"label": "unrelated"}"""

# ==================== AGENT 2: Sentiment Analyzer ====================
sentiment_system = """You are SentimentAgent.
Analyze the sentiment of the review text (English or Traditional Chinese, emojis, links allowed).
Output EXACTLY this JSON and nothing else:
{"sentiment": "positive"} or {"sentiment": "negative"} or {"sentiment": "neutral"}"""

# ==================== AGENT 3: Combiner (final validator) ====================
combiner_system = """You are CombinerAgent (final decision maker).
You receive:
- Title, URL, Text
- label from FactCheckerAgent
- sentiment from SentimentAgent

Keep the values unless there is an obvious error, then fix it.
Output EXACTLY this JSON and nothing else:
{"label": "real", "sentiment": "positive"}"""


In [8]:
# ========================== CELL 4: LOAD DATA & RUN 3 AGENTS ==========================
# Upload your data.jsonl to /content/data.jsonl first
data_path = "/content/drive/MyDrive/Czech/M12_final_v1_to_dataset.jsonl"

reviews = []
with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.strip():
            reviews.append(json.loads(line))
        if len(reviews) >= 100:          # stop after we have 100 valid records
            break

print(f"Loaded {len(reviews)} reviews (limited to first 100)")

results = []

for row in tqdm(reviews, desc="Processing with 3 Agents"):
    package_id = row.get("package_id", "")
    review_id = row.get("review_id", "")
    cid = row.get("cid", "")
    text = row.get("text", "")
    url = row.get("url") if row.get("url") is not None else "None"
    title = row.get("title") if row.get("title") is not None else "None"

    if not text.strip():
        label = "unrelated"
        sentiment = "neutral"
    else:
        # AGENT 1
        fact_user = f"Title: {title}\nURL: {url}\nText: {text}"
        fact_raw = query_agent(fact_system, fact_user)
        try:
            fact_json = json.loads(fact_raw)
            label = fact_json.get("label", "unrelated")
        except:
            label = "unrelated"

        # AGENT 2
        sent_user = f"Text: {text}"
        sent_raw = query_agent(sentiment_system, sent_user)
        try:
            sent_json = json.loads(sent_raw)
            sentiment = sent_json.get("sentiment", "neutral")
        except:
            sentiment = "neutral"

        # AGENT 3 - Combiner (final check)
        combiner_user = f"""Title: {title}
URL: {url}
Text: {text}
Fact label: {label}
Sentiment: {sentiment}
Output final JSON now."""

        final_raw = query_agent(combiner_system, combiner_user, max_new=100)
        try:
            final_json = json.loads(final_raw)
            label = final_json.get("label", label)
            sentiment = final_json.get("sentiment", sentiment)
        except:
            pass  # keep previous values

    results.append({
        "package_id": package_id,
        "review_id": review_id,
        "cid": cid,
        "url": url if url != "None" else "",
        "title": title if title != "None" else "",
        "text": text,
        "label": label,
        "sentiment": sentiment
    })


Loaded 100 reviews (limited to first 100)


Processing with 3 Agents: 100%|██████████| 100/100 [06:05<00:00,  3.66s/it]


In [9]:
# ========================== CELL 5: SAVE CSV ==========================
df = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/Czech/analyzed_reviews.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"DONE! CSV saved to {output_path}")
print(df.head())

# Optional: download
from google.colab import files
files.download(output_path)







DONE! CSV saved to /content/drive/MyDrive/Czech/analyzed_reviews.csv
  package_id review_id        cid url title  \
0         01  ptt_30_1  ptt_30_c1             
1         01  ptt_30_2  ptt_30_c2             
2         01  ptt_30_3  ptt_30_c3             
3         01  ptt_30_4  ptt_30_c4             
4         01  ptt_30_5  ptt_30_c5             

                                    text      label sentiment  
0               都說了，就別執著了。而且也沒意義，吃不了那麼高。  unrelated  negative  
1          走PD3.2，可能只有17PRO能吃滿也說不定，當專用即可  unrelated  negative  
2                 #1emPiow4 (MobileComm)  unrelated  negative  
3                              typec懂的都懂  unrelated  positive  
4  我上個月買Google Pixel Flex 67W雙C充電頭 支援AVS  unrelated  positive  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>